## Access StructureDCA coefficients and properties
Here is a small tutorial to access and understand formats of different StructureDCA properties (like fields `h`, couplings `J`, Frobenius norms, residue-residue distance matrix, contact map, ...).

### Step 1: Initialize StructureDCA
Let us first create a StructureDCA model using the sample data (MSA and 3D structure) from `../test_data/` for protein `6acv_A_29-94`.

In [ ]:
# Imports
import numpy as np
import matplotlib.pyplot as plt

from structuredca import StructureDCA
from structuredca.sequence import AminoAcid

In [ ]:
# Initialize StructureDCA model
sdca = StructureDCA(
    msa_path='../test_data/6acv_A_29-94.fasta',
    pdb_path='../test_data/6acv_A_29-94.pdb',
    chains='A',
    use_contacts_plddt_filter=False, # use only if 3D structure is an AlphaFold model (or similar) to remove low pLDDT regions from contacts
    num_threads=8,
    verbose=True,
)

### Step 2: StructureDCA input and DCA model dimensions
Access the StructureDCA model target sequence and model dimensions `L` (target sequence length) and `A=21` (number of possible states).

In [ ]:
# We can access StructureDCA target sequence of the MSA
target_sequence = sdca.target_sequence # As a Sequence object
print(target_sequence)

# Here is the target sequence as a Python string (str)
target_sequence_str = target_sequence.sequence
print(target_sequence_str)

In [ ]:
# Here is the DCA model dimensions
L = sdca.msa_length # Length of the target sequence in the MSA (=len(target_sequence_str))
A = sdca.n_states # Number of states (always 21: 20 Amino Acids and 1 gap)
print(f"StructureDCA dimensions: [L={L}, A={A}]")

In [ ]:
# Amino Acids indexation (to access DCA h and J coefficients)
#    * AAs are indexed by alphabetic order (index A: 0, index C: 1, index D: 2, ...)
#    * Gap character '-' is the last index 20
all_states_ordered = [aa for aa in AminoAcid.get_all()] + [AminoAcid.get_gap()]
for state in all_states_ordered:
    print(state)

# You can access the index of an Amino Acid from its one letter code with
print(f"\nIndex of 'H' is: {AminoAcid('H').id}")

### Step 3: DCA fields h and couplings J
Access DCA fields `h` and couplings `J` coefficients.
`h` and `J` coefficients are in NumPy `np.float32` type and can be easely accessed.

In [ ]:
# Access DCA fields h

# Fields h is a NumPy np.ndarray of shape [L, A=21]
h = sdca.h
print(f" * fields h: type={type(h)}")
print(f" * fields h: dtype={h.dtype}")
print(f" * fields h: shape={h.shape}\n")

# Coefficient h[i, a] represents "how happy" is Amino Acid a is at position i
# For instance, here is h for Proline ('P', id=12) at position 5
h_ia = h[5, 12]
print(f" * h[i=5, a=12 ('P')]: {h_ia:.4f}")

In [ ]:
# Access DCA couplings J
# WARNING: StructureDCA usually deals with very sparse coefficents J => most of these coefficients are zeros
# For computational-time and -memory efficiency, J is implemented as an Object SparseJ and not a standard np.ndarray

# Couplings J is a SparseJ tensor of shape [L, L, A=21, A=21]
J = sdca.J
print(f" * couplings J: type={type(J)}")
print(f" * couplings J: dtype={J.dtype}")
print(f" * couplings J: shape={J.shape}\n")

# You can access a J[i, j] (21, 21)-matrix (which is a standard np.ndarray)
J_ij = J[13, 14]
print(f" * couplings J_ij block: type={type(J_ij)}")
print(f" * couplings J_ij block: dtype={J_ij.dtype}")
print(f" * couplings J_ij block: shape={J_ij.shape}\n")

# But you CAN NOT directly access a single coefficient
#J_ijab = J[13, 14, 0, 1] # -> ERROR

# Rather you first need to access the J_ij (21, 21)-matrix
# Coefficient J[i, j, a, b] represents "how fit" is a pair of Amino Acids a and b at respective position i and j
# For instance, here is J for Amino Acid 'A' (index 0) at position 13 and Amino Acid 'C' (index 1) at positon 14
J_ijab = J[13, 14][0, 1]
print(f" * J[i=13, j=14, a=0 ('A'), b=1 ('C')]: {J_ijab:.4f}\n")

# You can also transform J to a standard np.ndarray (note that this can be very RAM intensive if L is very large)
J_numpy = J.to_numpy()
print(f" * couplings J (numpy): type={type(J_numpy)}")
print(f" * couplings J (numpy): dtype={J_numpy.dtype}")
print(f" * couplings J (numpy): shape={J_numpy.shape}")

# Alternatively you can directly ask StructureDCA to create J as a standard np.ndarray (and not a SparseJ Object)
# Set use_sparse_J=False at StructureDCA initialization as:
# sdca = StructureDCA(..., use_sparse_J=False)

### Step 4: Access other StructureDCA properties
Access other StructureDCA properties like Frobenius norms or distance matrix and so on.

**Note**: if 3D structure and MSA target sequence do not perfecty match, distance matrix, contact map, RSA array (and other structural properties) are mapped to MSA positions (by pairwise sequence alignment).
Those are thus of shape (L, L) or L independently of the length of the corresponding chain in the 3D structure.

In [ ]:
# Frobenius Norm
F = sdca.dca_model.frobenius_norm() # np.ndarray: ((L, L), np.float32)
print(f" * couplings Frobenius norms F: type={type(F)}")
print(f" * couplings Frobenius norms F: dtype={F.dtype}")
print(f" * couplings Frobenius norms F: shape={F.shape}\n")

# Number of sequences in the MSA
n = sdca.dca_model.msa_depth # (int) after MSA pre-processing step (like remove redundent sequences): only counting the sequences used to infer the DCA model
n_init = sdca.dca_model.msa_initial_depth # (int) before MSA pre-processing (all sequence in the MSA raw file)
n_eff = sdca.dca_model.Neff # (float) number of effective sequences = SUM_{i=1, ..., msa_depth} weight(sequence_i)
print(f" * Ntot (pre-processed): {n}")
print(f" * Ntot (initial): {n_init}")
print(f" * Neff : {n_eff}\n")

# Alignment between:
# * target sequence of the MSA
# * target chain in the 3D structure
# NOTE: these do not always perfectly match
align = sdca.alignment
align.show()
print("")

# RSA and pLDDT array by position
rsa = sdca.rsa_array # np.ndarray: ((L,), np.float32): array of Relative Solvent Accessibility (RSA) mapped to MSA positions
plddt = sdca.plddt_array # np.ndarray: ((L,), np.float32): array of pLDDT scores mapped to MSA positions (is Structure is an AlphaFold model or similar)
print(f" * RSA-array: shape={rsa.shape}")
print(f" * pLDDT-array: shape={plddt.shape}")

# Distance Matrix
dist = sdca.distance_matrix # np.ndarray: ((L, L), np.float32): matrix of residue-residue distances mapped to MSA positions
print(f" * distance-matrix: shape={dist.shape}")

# Contact Map (defined by distance matrix and parameter 'distance_cutoff')
contacts = sdca.contacts # np.ndarray ((L, L), np.bool_): matrix of residue-residue contacts mapped to MSA positions
print(f" * contacts-matrix: shape={contacts.shape}\n")

# Some metrics
n_contacts = sdca.dca_model.n_contacts() # (int) total number of contacts (count twice (i, j) and (j, i) but not diagonal (i, i))
avg_contacts = sdca.dca_model.avg_contacts() # (float) average number of contacts by position
contacts_ratio = sdca.dca_model.contacts_ratio() # (float) ratio of effective contacts among all possible contacts
print(f" * n_contacts: {n_contacts}")
print(f" * avg_contacts: {avg_contacts}")
print(f" * contacts_ratio: {contacts_ratio}")


Now let us just plot some of the obtained values to see if everything is coherent.

In [ ]:
# Show RSA and pLDDT arrays
plt.title("RSA and pLDDT")
plt.axhline(0.0, linestyle="--", color="black")
plt.axhline(100.0, linestyle="--", color="black")
plt.plot(list(range(L)), rsa, label="RSA")
plt.plot(list(range(L)), plddt, label="pLDDT")
plt.xticks(list(range(L)), target_sequence_str, fontsize=6)
plt.legend()
plt.show()


In [ ]:
# Show distance Matrix
plt.imshow(dist)
plt.title("Distance matrix")
plt.colorbar()
plt.yticks([])
plt.xticks([])
plt.show()


In [ ]:
# Show contacts map
plt.title("Contacts matrix")
plt.imshow(contacts, cmap="Greys")
plt.yticks([])
plt.xticks([])
plt.show()


In [ ]:
# Show Frobenius Norms
plt.title("Frobenius Norms")
plt.imshow(F, cmap='Blues')
plt.colorbar()
plt.xticks([])
plt.yticks([])
